# TIMELYMT P3-GLOBAL RESEARCH LAB

> **TRAIN + DEV ONLY**  
> **TEST UNTOUCHED**  
> **STOP BEFORE TEST**

P3_GLOBAL extends P2 with a fixed talk-level PreparedContext representation used only by the policy. This lab makes no claim of improvement.

## 1. OPERATOR CONFIGURATION

Edit this cell only. All consequential stages are disabled by default.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/MinhCYB/TimelyMT.git'
REPOSITORY_REF = 'main'
UPSTREAM_CHECKPOINT_DATASET_REF = 'iteams24/timelymt-research-checkpoints'
P3_CHECKPOINT_DATASET_REF = 'iteams24/timelymt-p3-global-checkpoints'
SELECTED_DEV_TALK = 'ted-sims-witherspoon-ai-climate'
EMPTY_CONTEXT_DEV_TALK = 'ted-jeff-dean-ai-smart'
SINGLE_THRESHOLD = 0.50
RUN_ENVIT5_SMOKE = False
RUN_TRAIN_P3 = False
FORCE_RETRAIN_P3 = False
PUBLISH_P3_CHECKPOINT = False
RUN_SINGLE_DEV_ROLLOUT = False
RUN_EMPTY_CONTEXT_ROLLOUT = False
RUN_FULL_DEV_ROLLOUT = False
RUN_DEV_EVALUATION = False

WORKING_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
REPO_ROOT = WORKING_ROOT / 'TimelyMT'
SRC_ROOT = REPO_ROOT / 'src'
CONFIG_PATH = REPO_ROOT / 'configs/experiments/policy-p3-global.json'
TRANSLATOR_CONFIG = REPO_ROOT / 'configs/translator/envit5.json'
PREPARED_MANIFEST = REPO_ROOT / 'data/prepared_context/manifest.json'
PSEUDO_TRAIN = REPO_ROOT / 'data/policy/pseudo_labels/train'
DOWNLOAD_ROOT = WORKING_ROOT / 'timelymt-p3-downloads'
P3_PACKAGE_ARCHIVE = WORKING_ROOT / 'timelymt-p3-global-checkpoint.tar.gz'
ARTIFACT_EXPORT_ROOT = WORKING_ROOT / 'timelymt-p3-global-artifacts'


## 2. KAGGLE / GPU ENVIRONMENT

Inspect the runtime before loading models. CPU setup remains valid, but current EnViT5 rollout expects CUDA.

In [ ]:
import torch
print(f'Python: {sys.version.split()[0]}')
print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for index in range(torch.cuda.device_count()):
    print(f'GPU {index}: {torch.cuda.get_device_name(index)}')
input_root = Path('/kaggle/input')
print(f'Kaggle input roots: {[str(path) for path in input_root.iterdir()] if input_root.exists() else []}')
if not torch.cuda.is_available():
    print('WARNING: CPU setup is supported, but EnViT5 rollout currently expects CUDA.')


## 3. REPOSITORY SETUP

Refresh the selected ref so this lab never silently runs a stale checkout. The repository `src` directory is authoritative.

In [ ]:
if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', REPOSITORY_REF, '--single-branch', REPOSITORY_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', REPOSITORY_REF], cwd=REPO_ROOT, check=True)
    subprocess.run(['git', 'checkout', REPOSITORY_REF], cwd=REPO_ROOT, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', REPOSITORY_REF], cwd=REPO_ROOT, check=True)
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
os.environ['PYTHONPATH'] = str(SRC_ROOT)
for command in (['git', 'rev-parse', 'HEAD'], ['git', 'log', '-1', '--oneline'], ['git', 'status', '--short']):
    subprocess.run(command, cwd=REPO_ROOT, check=True)
if not CONFIG_PATH.is_file() or not PREPARED_MANIFEST.is_file():
    raise FileNotFoundError('P3 source/config/prepared-context contract is incomplete')


## 4. DEPENDENCY / HF CACHE SETUP

Reuse Kaggle's CUDA-enabled PyTorch. Hugging Face assets use an explicit external cache and are never checkpoint artifacts.

In [ ]:
HF_CACHE = WORKING_ROOT / 'huggingface-cache'
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_CACHE / 'hub')
print(f'HF cache: {Path(os.environ["HF_HOME"]).resolve()}')
from timelymt.research.p3_checkpointing import (
    build_p3_package, discover_p3_candidates, discover_upstream_supervision_candidates,
    local_p3_checkpoint, repository_identity, resolve_local_conflict, restore_p3_candidate,
    restore_upstream_supervision,
)
from timelymt.research.policy_p3_global import prepared_manifest_fingerprint
from timelymt.research.policy_p3_global_runner import p3_runtime
from timelymt.research.policy_v2 import validate_v1_supervision


## 5. RESTORE FROZEN UPSTREAM TRAIN SUPERVISION

Restore only frozen V1 TRAIN pseudo-labels. Prefer a mounted Kaggle Dataset and use the Kaggle CLI only when no mounted package is available. Inspect source, manifest, and row count.

In [ ]:
def kaggle_download(dataset_ref, destination):
    destination.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(['kaggle', 'datasets', 'download', '-d', dataset_ref, '-p', str(destination), '--unzip'], text=True, capture_output=True)
    if result.returncode:
        print(result.stderr.strip() or result.stdout.strip())
        return False
    return True

def restore_upstream_if_needed():
    try:
        manifest, rows = validate_v1_supervision(PSEUDO_TRAIN, 'train')
        return {'restored': False, 'manifest': manifest, 'rows': rows, 'source': {'mode': 'already-local', 'package_root': REPO_ROOT}}
    except RuntimeError:
        pass
    for root, mode in ((Path('/kaggle/input'), 'mounted'), (DOWNLOAD_ROOT / 'upstream', 'downloaded')):
        if mode == 'downloaded':
            shutil.rmtree(root, ignore_errors=True)
            if not kaggle_download(UPSTREAM_CHECKPOINT_DATASET_REF, root):
                continue
        for candidate in discover_upstream_supervision_candidates(root):
            try:
                restore_upstream_supervision(candidate['package_root'], REPO_ROOT)
                manifest, rows = validate_v1_supervision(PSEUDO_TRAIN, 'train')
                return {'restored': True, 'manifest': manifest, 'rows': rows, 'source': candidate}
            except RuntimeError:
                continue
    raise RuntimeError('Required frozen V1 TRAIN supervision is absent and could not be restored.')

UPSTREAM_STATUS = restore_upstream_if_needed()
print(f"source mode: {UPSTREAM_STATUS['source']['mode']}")
print(f"package root: {UPSTREAM_STATUS['source']['package_root']}")
print(f"TRAIN manifest: {PSEUDO_TRAIN / 'manifest.json'}")
print(f"TRAIN row count: {len(UPSTREAM_STATUS['rows'])}")


## 6. RESTORE P3 CHECKPOINT

The official discovery and compatibility validator accepts raw archives or Kaggle-expanded packages. No checkpoint is trained when none is found.

In [ ]:
P3_RESTORE_STATUS = {'found': False, 'compatible': False, 'completed_stage': 'NONE', 'created_at': None, 'checkpoint_sha256': None, 'reachable': False}
persistent_root = DOWNLOAD_ROOT / 'p3'
mounted_candidates = discover_p3_candidates(Path('/kaggle/input'), config_path=CONFIG_PATH, manifest_path=PREPARED_MANIFEST)
if mounted_candidates:
    candidates = mounted_candidates
    P3_RESTORE_STATUS['reachable'] = True
else:
    shutil.rmtree(persistent_root, ignore_errors=True)
    reachable = kaggle_download(P3_CHECKPOINT_DATASET_REF, persistent_root)
    candidates = discover_p3_candidates(persistent_root, config_path=CONFIG_PATH, manifest_path=PREPARED_MANIFEST) if reachable else []
    P3_RESTORE_STATUS['reachable'] = reachable
if candidates:
    selected = candidates[0]
    local = local_p3_checkpoint(REPO_ROOT, config_path=CONFIG_PATH, manifest_path=PREPARED_MANIFEST)
    action = resolve_local_conflict(local, selected['metadata'])
    if action == 'conflict':
        raise RuntimeError('Local and persistent valid P3 checkpoints differ with ambiguous ordering.')
    if action == 'restore-persistent':
        restore_p3_candidate(selected['path'], REPO_ROOT, config_path=CONFIG_PATH, manifest_path=PREPARED_MANIFEST)
    P3_RESTORE_STATUS.update({'found': True, 'compatible': True, 'action': action, **selected['metadata']})
P3_LOCAL = local_p3_checkpoint(REPO_ROOT, config_path=CONFIG_PATH, manifest_path=PREPARED_MANIFEST)
P3_CHECKPOINT_VALID = P3_LOCAL is not None
for key in ('found', 'compatible', 'completed_stage', 'created_at', 'checkpoint_sha256'):
    print(f'{key}: {P3_RESTORE_STATUS.get(key)}')


## 7. P3 SESSION STATUS

Review this compact block before enabling any manual stage.

In [ ]:
prepared = json.loads(PREPARED_MANIFEST.read_text(encoding='utf-8'))
identity = repository_identity(REPO_ROOT)
runtime = p3_runtime(CONFIG_PATH)
gpu_name = torch.cuda.get_device_name(runtime['encoder_device']) if runtime['encoder_device'].type == 'cuda' else None
print('P3 SESSION STATUS')
print(f"Repository:\n  commit: {identity['repo_commit']}\n  dirty: {identity['working_tree_dirty']}")
print(f"Upstream supervision:\n  restored: YES\n  TRAIN rows: {len(UPSTREAM_STATUS['rows'])}")
print(f"Prepared context:\n  manifest fingerprint: {prepared_manifest_fingerprint(PREPARED_MANIFEST)}\n  TRAIN context talks: {sum(x['split'] == 'train' and bool(x.get('sources')) for x in prepared['pools'])}\n  DEV context talks: {sum(x['split'] == 'dev' and bool(x.get('sources')) for x in prepared['pools'])}")
print(f"P3 persistent dataset:\n  ref: {P3_CHECKPOINT_DATASET_REF}\n  reachable: {'YES' if P3_RESTORE_STATUS['reachable'] else 'NO'}")
print(f"P3 checkpoint:\n  found: {'YES' if P3_RESTORE_STATUS['found'] else 'NO'}\n  compatible: {'YES' if P3_CHECKPOINT_VALID else 'NO'}\n  stage: {P3_RESTORE_STATUS['completed_stage']}\n  created_at: {P3_RESTORE_STATUS['created_at']}")
print(f"TRAIN required: {'YES' if FORCE_RETRAIN_P3 or not P3_CHECKPOINT_VALID else 'NO'}")
print(f"Runtime:\n  encoder device: {runtime['encoder_device']}\n  policy device: {runtime['policy_device']}\n  GPU name: {gpu_name}")


## 8. ENVIT5 SMOKE TEST

This first-class stage precedes every DEV rollout. It uses only a synthetic string, never DEV data. Inspect tokenizer diagnostics before model loading.

In [ ]:
ENVIT5_SMOKE_PASSED = False
if not RUN_ENVIT5_SMOKE:
    print('ENVIT5 SMOKE: skipped intentionally (RUN_ENVIT5_SMOKE=False).')
else:
    from timelymt.translator.envit5 import load_config, tokenizer_diagnostics
    print(json.dumps(tokenizer_diagnostics(load_config(TRANSLATOR_CONFIG)), indent=2, sort_keys=True))
    subprocess.run([sys.executable, '-m', 'timelymt.translator.cli', '--text', 'A tiny synthetic English string.', '--device', 'cuda', '--config', str(TRANSLATOR_CONFIG)], cwd=REPO_ROOT, check=True)
    ENVIT5_SMOKE_PASSED = True
    print('ENVIT5 SMOKE: PASS')

def require_smoke_pass():
    if not ENVIT5_SMOKE_PASSED:
        raise RuntimeError('EnViT5 smoke must PASS before a DEV rollout. Enable RUN_ENVIT5_SMOKE and run this section first.')


## 9. P3 CHECKPOINT INSPECTION

Use the official inspection rather than reinterpreting checkpoint internals. Review identity, representation, labels, and runtime provenance.

In [ ]:
if P3_CHECKPOINT_VALID:
    subprocess.run([sys.executable, '-m', 'timelymt.research.cli', 'inspect-p3-checkpoint'], cwd=REPO_ROOT, check=True)
else:
    print('No compatible P3 checkpoint is available for inspection.')


## 10. MANUAL TRAIN P3

Training is disabled by default. A valid checkpoint blocks training unless forced intentionally.

In [ ]:
if RUN_TRAIN_P3:
    if P3_CHECKPOINT_VALID and not FORCE_RETRAIN_P3:
        raise RuntimeError('A compatible P3 checkpoint already exists. Set FORCE_RETRAIN_P3=True intentionally to retrain.')
    subprocess.run([sys.executable, '-m', 'timelymt.research.cli', 'train-p3'], cwd=REPO_ROOT, check=True)
    P3_LOCAL = local_p3_checkpoint(REPO_ROOT, config_path=CONFIG_PATH, manifest_path=PREPARED_MANIFEST)
    P3_CHECKPOINT_VALID = P3_LOCAL is not None
    subprocess.run([sys.executable, '-m', 'timelymt.research.cli', 'inspect-p3-checkpoint'], cwd=REPO_ROOT, check=True)
elif not P3_CHECKPOINT_VALID:
    print('No compatible P3 checkpoint exists. To train manually set RUN_TRAIN_P3=True, then run this section.')
else:
    print('TRAIN disabled: existing compatible P3 checkpoint is preserved.')


## 11. MANUAL P3 CHECKPOINT PUBLICATION

Publication validates the checkpoint and targets only the P3 Dataset. It does not alter upstream supervision, package HF cache, or package TEST.

In [ ]:
def publish_p3_checkpoint(*, stage='TRAINED'):
    if not PUBLISH_P3_CHECKPOINT:
        raise RuntimeError('Publishing is disabled. Set PUBLISH_P3_CHECKPOINT=True intentionally.')
    build_p3_package(REPO_ROOT, P3_PACKAGE_ARCHIVE, config_path=CONFIG_PATH, manifest_path=PREPARED_MANIFEST, stage=stage)
    upload = WORKING_ROOT / 'p3-checkpoint-upload'
    shutil.rmtree(upload, ignore_errors=True)
    upload.mkdir()
    shutil.copy2(P3_PACKAGE_ARCHIVE, upload / P3_PACKAGE_ARCHIVE.name)
    (upload / 'dataset-metadata.json').write_text(json.dumps({'title': 'TimelyMT P3 GLOBAL Checkpoints', 'id': P3_CHECKPOINT_DATASET_REF, 'licenses': [{'name': 'other'}]}, indent=2), encoding='utf-8')
    subprocess.run(['kaggle', 'datasets', 'version', '-p', str(upload), '-m', f'P3_GLOBAL {stage}', '--dir-mode', 'zip'], check=True)
    subprocess.run(['kaggle', 'datasets', 'status', '-d', P3_CHECKPOINT_DATASET_REF], check=True)
    subprocess.run(['kaggle', 'datasets', 'files', '-d', P3_CHECKPOINT_DATASET_REF], check=True)

if PUBLISH_P3_CHECKPOINT:
    publish_p3_checkpoint()
else:
    print('Publication skipped intentionally (PUBLISH_P3_CHECKPOINT=False).')


## 12. SINGLE DEV ROLLOUT - CONTEXT-BEARING TALK

Run the default Sims Witherspoon talk at threshold 0.50 only after smoke PASS and checkpoint validation.

In [ ]:
if RUN_SINGLE_DEV_ROLLOUT:
    require_smoke_pass()
    if not P3_CHECKPOINT_VALID:
        raise RuntimeError('A compatible P3 checkpoint is required for rollout.')
    subprocess.run([sys.executable, '-m', 'timelymt.research.cli', 'rollout-p3', '--split', 'dev', '--talk-id', SELECTED_DEV_TALK, '--thresholds', f'{SINGLE_THRESHOLD:.2f}', '--batch-size', '1'], cwd=REPO_ROOT, check=True)
else:
    print('Single DEV rollout skipped intentionally.')


## 13. INSPECT SINGLE DEV ARTIFACT

Current rollout artifacts do not capture every LISTEN decision. A later demo trace layer may add full timestep-level policy traces.

In [ ]:
single_artifact = REPO_ROOT / 'outputs/experiments/policy-p3-global/predictions/dev' / f'p3_global_{SINGLE_THRESHOLD:.2f}' / f'{SELECTED_DEV_TALK}.json'
if single_artifact.is_file():
    record = json.loads(single_artifact.read_text(encoding='utf-8'))
    commits = record.get('commits', [])
    print(json.dumps({'talk_id': record.get('talk_id'), 'strategy': record.get('strategy'), 'threshold': SINGLE_THRESHOLD, 'number_of_commits': len(commits), 'committed_source_spans': [item.get('source_text') for item in commits], 'translated_units': [item.get('translated_text') for item in commits], 'first_commit': commits[0] if commits else None, 'prepared_context': record.get('prepared_context')}, indent=2, ensure_ascii=False))
else:
    print(f'No single DEV artifact exists: {single_artifact}')


## 14. OPTIONAL EMPTY-CONTEXT DEV ROLLOUT

This demonstrates operation with an exact zero prepared-global embedding. It is not a scientific comparison by itself.

In [ ]:
if RUN_EMPTY_CONTEXT_ROLLOUT:
    require_smoke_pass()
    if not P3_CHECKPOINT_VALID:
        raise RuntimeError('A compatible P3 checkpoint is required for rollout.')
    subprocess.run([sys.executable, '-m', 'timelymt.research.cli', 'rollout-p3', '--split', 'dev', '--talk-id', EMPTY_CONTEXT_DEV_TALK, '--thresholds', f'{SINGLE_THRESHOLD:.2f}', '--batch-size', '1'], cwd=REPO_ROOT, check=True)
else:
    print('Empty-context DEV rollout skipped intentionally.')


## 15. FULL DEV THRESHOLD GRID

Run the frozen 0.30-0.70 threshold grid on DEV only after the single-talk inspection.

In [ ]:
if RUN_FULL_DEV_ROLLOUT:
    require_smoke_pass()
    if not P3_CHECKPOINT_VALID:
        raise RuntimeError('A compatible P3 checkpoint is required for rollout.')
    subprocess.run([sys.executable, '-m', 'timelymt.research.cli', 'rollout-p3', '--split', 'dev', '--thresholds', '0.30', '0.40', '0.50', '0.60', '0.70', '--batch-size', '1'], cwd=REPO_ROOT, check=True)
else:
    print('Full DEV threshold grid skipped intentionally.')


## 16. DEV EVALUATION

Reuse official BLEU, chrF2, AL, LAAL, and commit statistics only after complete DEV prediction coverage.

In [ ]:
if RUN_DEV_EVALUATION:
    subprocess.run([sys.executable, '-m', 'timelymt.research.cli', 'evaluate-p3', '--split', 'dev'], cwd=REPO_ROOT, check=True)
else:
    print('DEV evaluation skipped intentionally.')


## 17. RESULT TABLE

When metrics exist, inspect the grid without automatically selecting or freezing a winner.

In [ ]:
metrics_path = REPO_ROOT / 'outputs/experiments/policy-p3-global/metrics/dev/all.json'
if metrics_path.is_file():
    import pandas as pd
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    rows = []
    for strategy, value in metrics.items():
        rows.append({'strategy': strategy, 'threshold': float(strategy.rsplit('_', 1)[1]), 'BLEU': value.get('BLEU'), 'chrF2': value.get('chrF2'), 'AL': value.get('token_level_average_lagging'), 'LAAL': value.get('token_level_length_adaptive_average_lagging'), 'commit_count': value.get('commit_count')})
    display(pd.DataFrame(rows).sort_values('threshold'))
else:
    print('No DEV metrics artifact exists yet.')


## 18. ARTIFACT PATH SUMMARY

Copy selected artifacts unchanged to Kaggle working storage when needed.

In [ ]:
ARTIFACT_PATHS = {'checkpoint': REPO_ROOT / 'checkpoints/policy_p3_global/P3_GLOBAL.pt', 'checkpoint_metadata': REPO_ROOT / 'checkpoints/policy_p3_global/P3_GLOBAL.metadata.json', 'DEV_predictions': REPO_ROOT / 'outputs/experiments/policy-p3-global/predictions/dev', 'DEV_metrics': REPO_ROOT / 'outputs/experiments/policy-p3-global/metrics/dev'}
for name, path in ARTIFACT_PATHS.items():
    print(f'{name}: {path}')

def copy_selected_artifacts(*names):
    for name in names:
        source = ARTIFACT_PATHS[name]
        destination = ARTIFACT_EXPORT_ROOT / name
        if source.is_dir():
            shutil.copytree(source, destination, dirs_exist_ok=True)
        else:
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
    print(f'Copied unchanged artifacts to {ARTIFACT_EXPORT_ROOT}')


# 19. STOP BEFORE TEST

## STOP BEFORE TEST

No held-out TEST command is exposed by this notebook.